In [1]:
import os
import openai

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file
openai.api_key = os.environ['OPENAI_API_KEY']

In [2]:
from langchain_openai import OpenAIEmbeddings
from langchain.vectorstores import DocArrayInMemorySearch
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain.schema.runnable import RunnableMap
from langchain.schema.output_parser import StrOutputParser

In [3]:
#!pip install docarray

In [4]:
vectorstore = DocArrayInMemorySearch.from_texts(
    [
    "El sol es una estrella.",
    "Los delfines son mamíferos.",
    "La Torre Eiffel está en París.",
    "Las ballenas azules son los animales más grandes del planeta.",
    "El agua cubre el 70% de la superficie de la Tierra.",
    "El ajedrez es un juego de estrategia muy antiguo.",
    "Las medusas existen desde hace más de 500 millones de años.",
    "El español es el segundo idioma más hablado en el mundo.",
    "Las hormigas pueden cargar hasta 50 veces su propio peso.",
    "El café se originó en Etiopía.",
    "El Sahara es el desierto más grande del mundo.",
    "El cerebro humano tiene alrededor de 86 mil millones de neuronas.",
    "La Gran Muralla China mide más de 21,000 kilómetros.",
    "Los gatos tienen 32 músculos en cada oreja.",
    "Los koalas duermen hasta 22 horas al día."
    ],
    embedding=OpenAIEmbeddings()
)
retriever = vectorstore.as_retriever()

In [5]:
retriever.invoke ("Cuanto duermen los Koalas?")

[Document(page_content='Los koalas duermen hasta 22 horas al día.'),
 Document(page_content='Los delfines son mamíferos.'),
 Document(page_content='Las hormigas pueden cargar hasta 50 veces su propio peso.'),
 Document(page_content='Las ballenas azules son los animales más grandes del planeta.')]

In [6]:
retriever.invoke("Que hay en Paris")

[Document(page_content='La Torre Eiffel está en París.'),
 Document(page_content='El sol es una estrella.'),
 Document(page_content='El Sahara es el desierto más grande del mundo.'),
 Document(page_content='El español es el segundo idioma más hablado en el mundo.')]

In [7]:
template = """Responda la pregunta basándose únicamente en el siguiente contexto:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

In [8]:
model = ChatOpenAI()
output_parser = StrOutputParser()

chain = RunnableMap({
    "context": lambda x: retriever.invoke(x["question"]),
    "question": lambda x: x["question"]
}) | prompt | model | output_parser

In [9]:
chain.invoke({"question": "Que se puede hacer en Paris"})

'Se puede visitar la Torre Eiffel.'